# Collecting draft-night comments

`04_match_threads.ipynb` attributed threads to picks. This notebook fetches
the comment bodies for a selected subset of those threads from Arctic Shift.

Output is one JSON file per thread in `data/raw/reddit_comments/`, keyed by
post ID, so collection is resumable: a rerun skips every thread already on
disk. Sentiment scoring consolidates the files in `06_score_sentiment.ipynb`.

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE = "https://arctic-shift.photon-reddit.com"
COMMENTS_DIR = Path("..") / "data" / "raw" / "reddit_comments"
COMMENTS_DIR.mkdir(parents=True, exist_ok=True)

matched = pd.read_csv("../data/processed/matched_threads.csv")
print(f"matched thread-pick pairs: {len(matched)}")

matched thread-pick pairs: 16677


### Selecting which threads to fetch

Fetching all 16,677 matched threads is neither necessary nor honest. Three
filters, each with its accounting printed below:

1. **Tier preference.** Surname-only matches are only trusted when a pick has
   no full-name thread at all. The cautionary example is Green Bay 2021: the
   "Aaron Rodgers Discussion Megathread" (1,861 comments of trade drama)
   surname-matches Amari Rodgers, whose actual selection threads all match at
   the full-name tier. Preferring the full tier removes that entire class of
   celebrity-surname contamination.
2. **Zero-comment threads** carry no sentiment and cost one request each.
3. **Top 5 threads per pick** by comment count. The tail beyond five is tiny
   threads; the mass kept is printed so the cap is a documented trade-off,
   not a silent truncation.

In [2]:
key = ["season", "team", "pick"]

n_full = (matched[matched.match_tier == "full"]
          .groupby(key)["post_id"].count().rename("n_full"))
sel = matched.merge(n_full, on=key, how="left")
sel = sel[(sel.match_tier == "full") | (sel.n_full.isna())]
surname_only = sel[sel.match_tier == "surname"].groupby(key).ngroups
print(f"tier preference: {len(sel)} threads over {sel.groupby(key).ngroups} picks "
      f"({surname_only} picks rely on surname-only matches)")

before = sel.groupby(key).ngroups
sel = sel[sel.num_comments > 0]
print(f"dropping zero-comment threads: {len(sel)} threads over "
      f"{sel.groupby(key).ngroups} picks "
      f"({before - sel.groupby(key).ngroups} picks lost — nothing to score)")

sel = (sel.sort_values("num_comments", ascending=False)
       .groupby(key).head(5)
       .sort_values(["season", "team", "pick"]))
kept_mass = sel.num_comments.sum() / matched.num_comments.sum()
print(f"top-5 cap: {len(sel)} threads to fetch, "
      f"{sel.num_comments.sum():,} comments upper bound, "
      f"{kept_mass:.1%} of all matched comment mass")

tier preference: 12978 threads over 1539 picks (22 picks rely on surname-only matches)
dropping zero-comment threads: 11100 threads over 1532 picks (7 picks lost — nothing to score)
top-5 cap: 5487 threads to fetch, 391,572 comments upper bound, 65.4% of all matched comment mass


### Fetching with a reaction window

Comments are fetched from thread creation to **48 hours** after it. The
question is how the fanbase reacted on draft night, not what it thinks in
hindsight once rookie camp footage exists — and the window also bounds the
cost of megathreads that stay active for weeks.

Pagination mirrors the thread harvest: fixed pages of 100 sorted ascending by
time, advancing `after` past each page. Three wrinkles specific to comments:

- `after` is **exclusive** and busy threads hold many comments in the same
  second, so the cursor advances to `max(created_utc) - 1` and relies on ID
  deduplication rather than risking same-second losses.
- If a page yields nothing new (a pathological second with >100 comments),
  the cursor force-advances by one second rather than looping forever.
- Arctic Shift enforces a **windowed rate limit** (HTTP 429 with an
  `x-ratelimit-reset` header). Blind exponential backoff wastes most of each
  window; a 429 instead pauses exactly until the advertised reset. At the
  observed budget (~20 requests/minute) the full harvest of ~9,400 requests
  takes several hours — the loop below is resumable, so it can simply be
  rerun until everything is on disk.

In [3]:
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4,
    backoff_factor=1.0,
    status_forcelist=[500, 502, 503, 504],
    allowed_methods=["GET"],
)))

PAGE_SIZE = 100
MAX_PAGES = 80
WINDOW_SECONDS = 48 * 3600

def rated_get(params, session):
    """GET that sleeps to the rate-limit reset horizon on 429."""
    while True:
        resp = session.get(f"{BASE}/api/comments/search",
                           params=params, timeout=60)
        if resp.status_code == 429:
            time.sleep(float(resp.headers.get("x-ratelimit-reset", 15)) + 1.0)
            continue
        resp.raise_for_status()
        return resp

def fetch_comments(post_id, thread_created, session):
    """All comments in a thread's first 48 hours, deduplicated by ID."""
    seen = {}
    cursor = thread_created - 1
    deadline = thread_created + WINDOW_SECONDS

    for _ in range(MAX_PAGES):
        resp = rated_get({
            "link_id": post_id,
            "after": cursor,
            "before": deadline,
            "limit": PAGE_SIZE,
            "sort": "asc",
            "fields": "id,body,score,created_utc,author",
        }, session)
        batch = resp.json()["data"]
        if not batch:
            break

        new = {c["id"]: c for c in batch if c["id"] not in seen}
        seen.update(new)

        if len(batch) < PAGE_SIZE:
            break
        newest = max(c["created_utc"] for c in batch)
        cursor = newest - 1 if new else cursor + 1

    return list(seen.values())

In [4]:
todo = sel.drop_duplicates("post_id")
done = fetched = failed = 0
t0 = time.time()

for t in todo.itertuples():
    out_path = COMMENTS_DIR / f"{t.post_id}.json"
    if out_path.exists():
        done += 1
        continue
    try:
        comments = fetch_comments(t.post_id, int(t.created_utc), session)
        out_path.write_text(json.dumps(comments))
        fetched += 1
    except Exception as exc:
        print(f"FAILED {t.post_id} ({t.team} {t.season} {t.pfr_player_name}): {exc}")
        failed += 1

    if (done + fetched) % 250 == 0:
        elapsed = time.time() - t0
        print(f"{done + fetched}/{len(todo)} threads "
              f"({fetched} fetched, {done} cached, {failed} failed, "
              f"{elapsed/60:.0f} min elapsed)", flush=True)

print(f"\ndone: {fetched} fetched, {done} cached, {failed} failed")


done: 0 fetched, 5487 cached, 0 failed


In [5]:
# What actually landed on disk, joined back to picks
counts = {p.stem: len(json.loads(p.read_text()))
          for p in COMMENTS_DIR.glob("*.json")}
sel["n_fetched"] = sel["post_id"].map(counts)

per_pick = (sel.dropna(subset=["n_fetched"])
            .groupby(key)["n_fetched"].sum())
print(f"threads on disk: {sel['n_fetched'].notna().sum()} / {len(sel)}")
print(f"comments fetched: {int(sel['n_fetched'].sum()):,}")
print(f"picks with any comments: {(per_pick > 0).sum()}")
print("\ncomments per pick:")
print(per_pick.describe().round(1).to_string())

threads on disk: 5487 / 5487
comments fetched: 402,999
picks with any comments: 1531

comments per pick:
count    1532.0
mean      263.1
std       318.5
min         0.0
25%        65.0
50%       144.5
75%       334.0
max      2304.0
